# 01 · Data exploration

Shapes, class balance, missing values, feature types and the train/calibration/test split of the two datasets. Produces the dataset tables used in the report:

- `results/datasets.csv`: one row per dataset
- `results/missing_values.csv`: missing cells per column (raw data)
- `results/splits.csv`: size and positive share of each split, for every seed

In [1]:
# Configuration
SEEDS = range(10)
DATASETS = ("spambase", "adult")
RECOMPUTE = True  # False: load the tables from results/ instead of recomputing them

In [2]:
import logging

import numpy as np
import pandas as pd

from calib.data import DATASETS as DATASET_SPECS
from calib.data import (
    Preprocessor, dataset_summary, fetch_raw, load_dataset, missing_mask,
    split_summary, stratified_split,
)
from calib.paths import RESULTS_DIR

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)
pd.set_option("display.precision", 4)

## Datasets

In [3]:
path = RESULTS_DIR / "datasets.csv"
if RECOMPUTE:
    datasets = pd.DataFrame([dataset_summary(name, seed=0) for name in DATASETS])
    datasets.to_csv(path, index=False)
else:
    datasets = pd.read_csv(path)
datasets.set_index("dataset").astype(object).T

calib.data: spambase: 4601 raw rows, 0 dropped (missing values), 4601 kept, 57 features, positive share 0.3940


calib.data: adult: 48842 raw rows, 3620 dropped (missing values), 45222 kept, 14 features, positive share 0.2478


dataset,spambase,adult
openml_version,1,2
n_raw,4601,48842
n_rows_with_missing,0,3620
n_rows,4601,45222
n_features,57,14
n_numeric,57,6
n_categorical,0,8
n_encoded,57,104
n_pos,1813,11208
pos_share,0.394,0.2478


## Missing values

Counted on the raw OpenML data, before any row is dropped.

In [4]:
path = RESULTS_DIR / "missing_values.csv"
if RECOMPUTE:
    rows = []
    for name in DATASETS:
        X_raw, _ = fetch_raw(name)
        counts = missing_mask(X_raw).sum()
        for column, n_missing in counts[counts > 0].items():
            rows.append({"dataset": name, "column": column, "n_missing": int(n_missing),
                         "share": n_missing / len(X_raw)})
    missing = pd.DataFrame(rows, columns=["dataset", "column", "n_missing", "share"])
    missing.to_csv(path, index=False)
else:
    missing = pd.read_csv(path)
missing

,dataset,column,n_missing,share
0,adult,workclass,2799,0.0573
1,adult,occupation,2809,0.0575
2,adult,native-country,857,0.0175


Are the rows with missing values different? Positive share among rows with and without missing values:

In [5]:
for name in DATASETS:
    X_raw, labels = fetch_raw(name)
    has_missing = missing_mask(X_raw).any(axis=1).to_numpy()
    is_pos = (labels == DATASET_SPECS[name]["positive"]).to_numpy()
    if has_missing.any():
        print(f"{name}: positive share {is_pos[has_missing].mean():.4f} in the {has_missing.sum()} rows "
              f"with missing values, {is_pos[~has_missing].mean():.4f} in the {(~has_missing).sum()} kept rows")

adult: positive share 0.1323 in the 3620 rows with missing values, 0.2478 in the 45222 kept rows


## Feature types

In [6]:
data = {name: load_dataset(name) for name in DATASETS}

for name, (X, y) in data.items():
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical = [c for c in X.columns if c not in numeric]
    print(f"{name}: {len(numeric)} numeric, {len(categorical)} categorical")

calib.data: spambase: 4601 raw rows, 0 dropped (missing values), 4601 kept, 57 features, positive share 0.3940


calib.data: adult: 48842 raw rows, 3620 dropped (missing values), 45222 kept, 14 features, positive share 0.2478


spambase: 57 numeric, 0 categorical
adult: 6 numeric, 8 categorical


Categorical columns of Adult and their number of levels (after dropping rows with missing values):

In [7]:
X_adult, _ = data["adult"]
categorical = X_adult.select_dtypes(exclude="number")
categorical.nunique().rename("n_levels").to_frame()

,n_levels
workclass,7
education,16
marital-status,7
occupation,14
relationship,6
race,5
sex,2
native-country,41


Numeric columns of Adult:

In [8]:
X_adult.select_dtypes(include="number").describe().T[["mean", "std", "min", "50%", "max"]]

,mean,std,min,50%,max
age,38.5479,13.2179,17.0,37.0,90.0
fnlwgt,189734.7343,105639.1951,13492.0,178316.0,1490400.0
education-num,10.1185,2.5529,1.0,10.0,16.0
capital-gain,1101.4303,7506.4301,0.0,0.0,99999.0
capital-loss,88.5954,404.9561,0.0,0.0,4356.0
hours-per-week,40.9380,12.0075,1.0,40.0,99.0


Spambase: all 57 features are numeric. Share of exact zeros and skewness, summarised over the 57 columns:

In [9]:
X_spam, _ = data["spambase"]
pd.DataFrame({
    "share_zeros": (X_spam == 0).mean(),
    "skewness": X_spam.skew(),
}).describe().loc[["min", "25%", "50%", "75%", "max"]]

,share_zeros,skewness
min,0.0000,1.5917
25%,0.7631,5.9570
50%,0.8370,9.7248
75%,0.9185,13.5838
max,0.9898,31.0621


## Train / calibration / test split

60 / 20 / 20 stratified split (`stratified_split`), recomputed for every seed.

In [10]:
path = RESULTS_DIR / "splits.csv"
if RECOMPUTE:
    tables = []
    for name, (X, y) in data.items():
        for seed in SEEDS:
            table = split_summary(y, stratified_split(y, rng=seed))
            tables.append(table.assign(dataset=name, seed=seed))
    splits = pd.concat(tables, ignore_index=True)[["dataset", "seed", "split", "n", "n_pos", "pos_share"]]
    splits.to_csv(path, index=False)
else:
    splits = pd.read_csv(path)
splits[splits["seed"] == 0]

,dataset,seed,split,n,n_pos,pos_share
0,spambase,0,train,2761,1088,0.3941
1,spambase,0,cal,919,362,0.3939
2,spambase,0,test,921,363,0.3941
30,adult,0,train,27133,6725,0.2479
31,adult,0,cal,9044,2241,0.2478
32,adult,0,test,9045,2242,0.2479


Do the split sizes depend on the seed? (They should not: the cut points depend only on the class counts.)

In [11]:
splits.groupby(["dataset", "split"], sort=False).agg(
    n_min=("n", "min"), n_max=("n", "max"),
    pos_share_min=("pos_share", "min"), pos_share_max=("pos_share", "max"),
)

n_min  n_max  pos_share_min  pos_share_max
dataset  split                                            
spambase train   2761   2761         0.3941         0.3941
         cal      919    919         0.3939         0.3939
         test     921    921         0.3941         0.3941
adult    train  27133  27133         0.2479         0.2479
         cal     9044   9044         0.2478         0.2478
         test    9045   9045         0.2479         0.2479

Width of the encoded feature matrix (Preprocessor fitted on each seed's training split):

In [12]:
widths = {
    name: sorted({
        len(Preprocessor().fit(X.iloc[stratified_split(y, rng=seed)[0]]).feature_names_)
        for seed in SEEDS
    })
    for name, (X, y) in data.items()
}
widths

{'spambase': [57], 'adult': [103, 104]}

## Observations

**Spambase.** 4,601 e-mails, 57 numeric features, 39.4% spam, no missing values. The features are word/character frequencies and capital-letter run lengths: the median column is 84% zeros and has skewness ≈ 10. Standardization rescales them but does not remove the skew; we keep the features as they are. 394 rows are exact duplicates of another row, so a few identical e-mails can fall in both training and test.

**Adult.** 48,842 raw rows; 3,620 (7.4%) have a missing value in `workclass`, `occupation` or `native-country` and are dropped, leaving 45,222 rows with 24.8% `>50K`. The dropped rows are not a random sample: only 13.2% of them are `>50K`, so dropping them slightly shifts the population (a limitation to state in the report). Six numeric features (`capital-gain` is very skewed, with 99,999 as a cap value) and eight categorical ones, one-hot encoded into 104 columns. `native-country = Holand-Netherlands` occurs in a single row, so on the seeds where that row is not in the training split (3 and 7) the encoding has 103 columns. 52 duplicate rows.

**Splits.** The stratified 60/20/20 split gives the same sizes for every seed (the cut points depend only on the class counts), and each split's positive share matches the full dataset to within 0.0002. The calibration sets have 919 rows (Spambase) and 9,044 rows (Adult), ten times more on Adult. That matters for isotonic regression, whose variance depends on the size of the calibration set.